# Weight experiment — Stage A (seed 42, 5 cấu hình)

- Qwen3-4B LoRA, 5M token/run, 625 optimizer step, ~8.000 token/step
- 5 cấu hình: natural, balanced, no_desc, tiki_heavy, title_heavy
- Chỉ seed 42 ở stage này; seed 123 chạy sau cho natural + candidate thắng
- Không chạy lại nếu `hyp_<config>_seed42.parquet` đã có

Mục tiêu: lấy tín hiệu paired trên cùng eval set, tiết kiệm quota.


In [ ]:
import os, subprocess, sys
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF','expandable_segments:True')
subprocess.run([sys.executable,'-m','pip','install','-q','-U','transformers>=4.51','peft','accelerate','huggingface_hub'], check=False)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'], capture_output=True)
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available(), 'count', torch.cuda.device_count())
if not torch.cuda.is_available():
    raise RuntimeError('Không có GPU')
p = torch.cuda.get_device_properties(0)
print('gpu', p.name, round(p.total_memory/1e9,1), 'GB')
if p.total_memory < 40e9:
    raise RuntimeError(f'Cần RTX Pro 6000 >=40GB, Kaggle đang cấp {p.name} {p.total_memory/1e9:.1f}GB. '
                       'Chọn Settings -> Accelerator -> GPU RTX Pro 6000 rồi Run All.')


In [ ]:
from huggingface_hub import snapshot_download
root = snapshot_download('ntquang0410/tiki-4model-eval', repo_type='dataset',
                         local_dir='/kaggle/working/tiki-4model-eval')
print('dataset', root)


In [ ]:
import subprocess, sys
from pathlib import Path
root = Path('/kaggle/working/tiki-4model-eval')
out = Path('/kaggle/working/weight_runs'); out.mkdir(parents=True, exist_ok=True)
configs = ['natural','balanced','no_desc','tiki_heavy','title_heavy']
for cfg in configs:
    hyp = out / f'hyp_{cfg}_seed42.parquet'
    if hyp.exists():
        print('skip existing', hyp); continue
    cmd = [sys.executable, str(root/'code/train_weight_gpu.py'),
           '--model','Qwen/Qwen3-4B',
           '--train-view', str(root/'data/weight_eval/train_view_4b.parquet'),
           '--eval-dir', str(root/'data/weight_eval'),
           '--configs', cfg, '--seeds','42',
           '--tokens','5000000','--steps','625',
           '--out', str(out), '--dtype','bf16','--micro-batch','8']
    print('RUN', cfg, flush=True)
    subprocess.run(cmd, check=True)
print('STAGE A DONE')


In [ ]:
# Chấm COMET ngay trên GPU sau khi train xong (không cần giữ 4B trong RAM)
import subprocess, sys
from pathlib import Path
root = Path('/kaggle/working/tiki-4model-eval')
out = Path('/kaggle/working/weight_runs')
subprocess.run([sys.executable,'-m','pip','install','-q','unbabel-comet','sacrebleu'], check=False)
subprocess.run([sys.executable, str(root/'code/score_comet_weight.py'),
                '--hyp-dir', str(out), '--out-dir', str(out), '--batch-size','8'], check=True)


In [ ]:
from pathlib import Path
for p in sorted(Path('/kaggle/working/weight_runs').glob('hyp_*.parquet')):
    print(p.name, p.stat().st_size)
for p in sorted(Path('/kaggle/working/weight_runs').glob('scores_*.parquet')):
    print(p.name, p.stat().st_size)
